# Imports and LLM links

In [1]:
import os
import jsonlines
from openai import OpenAI
import asyncio
import ast
from collections import Counter
from itertools import chain
import json
from pathlib import Path
import math


llm_base_url = "https://ki-toolbox.scc.kit.edu/api/v1"
llm_model = "kit.gpt-oss-120b"

try:
    llm_api_key = os.environ["KIT_AI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Umgebungs‑Variable KIT_AI_API_KEY fehlt") from exc



# Laden der Reden

In [2]:
legislatur = 21
alleReden = []
with jsonlines.open(f'../../data/speeches_{legislatur}.jsonl') as f:
    for line in f.iter():
        alleReden.append(line)

alleReden.sort(key = lambda x:x['date'])
len(alleReden)


8824

# Predefined Topics

In [3]:
topics = ['Zuwanderung und Flucht', 'Energiepolitik und Energiemanagement', 'Wirtschaftslage', 'Inflation', 'Gesundheitswesen', 'Bildung', 'Wohnen und Mieten', 'Verkehr und Mobilität', 'Soziale Ungleichheit', 'Digitalisierung', 'Rente und Altersvorsorge', 'Sicherheit und Terrorismus', 'Künstliche Intelligenz', 'Cybersicherheit', 'Globalisierung', 'Europäische Union', 'Forschung und Innovation', 'Klimaschutz', 'Arbeitsmarkt', 'Integration', 'Außenpolitik', 'Kultur und Kunst', 'Familienpolitik', 'Geschlechtergerechtigkeit', 'Sport und Freizeit', 'Konsumverhalten', 'Medien und Kommunikation', 'Rechtsextremismus', 'Demokratie und Partizipation', 'Jugendpolitik', 'Städteentwicklung', 'Landwirtschaft', 'Infrastrukturprojekte', 'Finanzpolitik', 'Krise und Katastrophenmanagement', 'Friedenspolitik', 'Tierschutz', 'Mittelstand und KMU', 'Bildungsgerechtigkeit', 'Tourismus', 'Kommunalpolitik', 'Ehrenamt und Zivilgesellschaft', 'Verteidigung', 'Sonstige Themen']

descriptions = ['Der Umgang mit Migration, Asyl und Integration ', 'Die Aufrechterhaltung und Ausbau von Erneuerbaren Energien sowie fossiler Energieträger', 'Die wirtschaftliche Stabilität ist zentrales Anliegen.', 'Bekämpfung und Steuerung der Inflation sind zentrale Anliegen ', 'Verbesserungen im Gesundheitssystem und die Bewältigung der Folgen der Pandemie stehen im Fokus.', 'Reformen und Investitionen im Bildungssektor, Förderung von Chancengleichheit und Erhöhung der Bildungsqualität', 'Wohnungsnot, Bauvorhaben und steigende Mietpreise stehen im Fokus', 'Der Ausbau nachhaltiger Verkehrsnetze und die Förderung von E-Mobilität sind wichtige Themen.', 'Maßnahmen zur Bekämpfung der sozialen Ungleichheit und zur Förderung der sozialen Gerechtigkeit.', 'Fortschritte in der Digitalisierung und die Förderung von digitaler Infrastruktur.', 'Sicherung der Renten und Anpassungen an demografische Veränderungen.', 'Innere Sicherheit und Maßnahmen gegen Terrorismus.', 'Regulierung und Integration von KI in den Alltag und die Arbeitswelt.', 'Schutz vor Cyberangriffen und Datenschutz.', 'Auswirkungen der Globalisierung auf die deutsche Wirtschaft und Gesellschaft.', 'Die Rolle Deutschlands in der EU und die Europawahlen.', 'Förderung von Wissenschaft und technologischen Innovationen.', 'Maßnahmen zum Schutz der Klima, Natur, Umwelt und  Biodiversität für eine nachhaltige Umwelt', 'Herausforderungen und Chancen im Arbeitsmarkt, insbesondere durch Automatisierung und KI.', 'Erfolgreiche Integration von Geflüchteten und Minderheiten in die Gesellschaft und den Arbeitsmarkt.', 'Deutschlands Rolle in der internationalen Politik und Beziehungen zu anderen Ländern.', 'Förderung von Kultur und Kunst sowie deren Zugang für alle Bevölkerungsschichten.', 'Unterstützung von Familien und Kinderbetreuung.', 'Förderung der Gleichstellung der Geschlechter.', 'Organisation und Förderung von Sportereignissen', 'Veränderungen im Konsumverhalten und deren Auswirkungen.', 'Veränderungen in der Medienlandschaft und die Rolle der sozialen Medien.', 'Bekämpfung von Rechtsextremismus und Rassismus.', 'Stärkung der demokratischen Strukturen und Bürgerbeteiligung.', 'Förderung von Jugendprojekten und Partizipationsmöglichkeiten für junge Menschen.', 'Nachhaltige Entwicklung und Planung von Städten.', 'Zukunft der Landwirtschaft und nachhaltige Agrarwirtschaft.', 'Ausbau und Modernisierung der deutschen Infrastruktur.', 'Maßnahmen zur Haushaltskonsolidierung und Steuerpolitik sowie Regulierung der Finanzmärkte', 'Vorbereitungen und Maßnahmen zur Bewältigung von Krisen.', 'Einsatz für internationale Friedensbemühungen.', 'Maßnahmen zum Schutz von Tieren und Tierrechten.', 'Unterstützung für kleine und mittlere Unternehmen.', 'Maßnahmen zur Förderung der Chancengleichheit im Bildungsbereich.', 'Förderung und nachhaltige Entwicklung des Tourismus.', 'Herausforderungen und Chancen auf kommunaler Ebene.', 'Unterstützung und Förderung des ehrenamtlichen Engagements.', 'Gewährleistung der nationale Sicherheit durch eine Kombination aus diplomatischen, militärischen und zivilen Maßnahmen sowie durch die Zusammenarbeit mit internationalen Partnern.', 'bspw. Geschäftsordnung etc. also alles was den oberen Themen nicht zuordnungsbar ist.']

# Functions for calling the LLM

In [4]:
def call_llm(messages, temperature: float = 0.0) -> str:
    client = OpenAI(api_key=llm_api_key, base_url=llm_base_url)
    try:
        # Send chat request
        resp = client.chat.completions.create(
            model=llm_model,
            messages=messages,
            temperature=temperature)
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return resp.choices[0].message.content.strip()



# Prompt - Beispielaufruf für eine Rede

In [5]:
# rede = alleReden[11111]
# 
# num_topics = 3
# topic_descriptions = "\n".join(
#     [f"[{topic}]: {desc}" for topic, desc in zip(topics, descriptions)]
# )
# 
# system_msg = f"""Du bist ein politischer Themenanalyst.
# Du bist darauf spezialisiert, Bundestagsreden präzise nach vordefinierten Themen zu klassifizieren.
# Die vordefinierten Themen mit einer kurzen Beschreibung findest du hier:
# {topic_descriptions}.
# 
# Bitte wähle nur aus diesen vordefinierten Themen. Keine Beschreibungen!"""
# 
# user_msg = f'''Ordne eine Rede aus dem deutschen Bundestag thematisch einer Kategorie zu. Hier sind die Kategorien als python-Liste {topics}. Gib ausschließlich die {num_topics} relevantesten Kategorien in dieser Liste zurück. Bitte als Output nur eine python-Liste. Keine Beschreibungen oder Erläuterungen!
# 
# Rede:
# {rede['text']}
# '''
# 
# messages=[
#     {
#         "role": "system",
#         "content": system_msg
#     },
#     {
#         "role": "user",
#         "content": user_msg
#     }
# ]
# 
# response = call_llm(messages,temperature=0.9)
# print(response)
#print(system_msg)

# Functions for Repeated Call for the same speech

In [6]:
#num_topics = 3

topic_descriptions = "\n".join(
    [f"[{topic}]: {desc}" for topic, desc in zip(topics, descriptions)]
)

system_msg = f"""Du bist ein politischer Themenanalyst.
Du bist darauf spezialisiert, Bundestagsreden präzise nach vordefinierten Themen zu klassifizieren.
Die vordefinierten Themen mit einer kurzen Beschreibung findest du hier:
{topic_descriptions}.

Bitte wähle nur aus diesen vordefinierten Themen. Keine Beschreibungen!"""

#########################################################

def sample_llm(rede,num_topics,num_samples):
    user_msg = f"""Ordne eine Rede aus dem deutschen Bundestag thematisch einer Kategorie zu. Hier sind die Kategorien als python-Liste

    {topics}. 

    Gib ausschließlich die {num_topics} relevantesten Kategorien in dieser Liste zurück. Bitte als Output nur eine python-Liste. Keine Beschreibungen oder Erläuterungen!

    Rede:
    {rede['text']}
    """

    messages=[
        {"role": "system","content": system_msg},
        {"role": "user","content": user_msg}
    ]

    all_cats = []
    for sample in range(num_samples):
        failed = 0
        try:
            response = call_llm(messages,temperature=0.9)
            print(response)
            cats = ast.literal_eval(response)
            all_cats.append(cats)
        except Exception as e:
            print(f"[Sample {sample+1}] Error: {e}")
            failed += 1
            continue
            
    return all_cats

#########################################################

async def sample_llm_async(rede,num_topics,num_samples):
    user_msg = f"""Ordne eine Rede aus dem deutschen Bundestag thematisch einer Kategorie zu. Hier sind die Kategorien als python-Liste

    {topics}. 

    Gib ausschließlich die {num_topics} relevantesten Kategorien in dieser Liste zurück. Bitte als Output nur eine python-Liste. Keine Beschreibungen oder Erläuterungen!

    Rede:
    {rede['text']}
    """

    messages=[
        {"role": "system","content": system_msg},
        {"role": "user","content": user_msg}
    ]

    tasks = [
        asyncio.to_thread(call_llm, messages, temperature=0.9)
        for _ in range(num_samples)
    ]
    
    raw_outputs = await asyncio.gather(*tasks)

    return [
        ast.literal_eval(output)
        for output in raw_outputs
    ]  


# Performance-Vergleich

In [7]:
# rede = alleReden[11111]
# num_samples = 3
# num_topics = 5
# result = sample_llm(rede,num_topics,num_samples)
# print(result)

In [8]:
# rede = alleReden[11111]
# num_samples = 3
# num_topics = 5
# result = await sample_llm_async(rede,num_topics,num_samples)
# print(result)


# The big loop

In [14]:
num_topics = 5
num_samples = 5
batch = 0
batch_size = 50

num_batches = math.ceil(len(alleReden) / batch_size)

output_dir = Path(f"llm_topic_annotations_{legislatur}")
output_dir.mkdir(exist_ok=True)

for batch in range(num_batches):
    output_file = output_dir / f"autosave_results_{batch}.json"

    # Skip already completed batches
    if output_file.exists():
        print(f"Skipping batch {batch + 1}/{num_batches}, already exists")
        continue

    start = batch * batch_size
    end = min(start + batch_size, len(alleReden))

    reden_subset = alleReden[start:end]

    print(f"Running batch {batch + 1}/{num_batches}: speeches {start}–{end - 1}")

    tasks = [
        sample_llm_async(rede, num_topics, num_samples)
        for rede in reden_subset
    ]

    results = await asyncio.gather(*tasks)

    themen_annotation_batch = {}

    for rede, result in zip(reden_subset, results):

        if isinstance(result, Exception):
            print(f"Error for speech {rede['id']}: {result}")
            continue
            
        flat = list(chain.from_iterable(result))
        counter = Counter(flat)
        total = sum(counter.values())
        counter = {
            k: v * num_topics / total
            for k, v in counter.items()
        }
        sorted_topics = dict(
            sorted(counter.items(), key=lambda x: x[1], reverse=True)
        )
        themen_annotation_batch[rede['id']] = sorted_topics
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(themen_annotation_batch, f, ensure_ascii=False, indent=2)

    print(f"Saved {output_file}")


Skipping batch 1/177, already exists
Skipping batch 2/177, already exists
Skipping batch 3/177, already exists
Skipping batch 4/177, already exists
Skipping batch 5/177, already exists
Skipping batch 6/177, already exists
Skipping batch 7/177, already exists
Skipping batch 8/177, already exists
Skipping batch 9/177, already exists
Skipping batch 10/177, already exists
Skipping batch 11/177, already exists
Skipping batch 12/177, already exists
Skipping batch 13/177, already exists
Skipping batch 14/177, already exists
Skipping batch 15/177, already exists
Skipping batch 16/177, already exists
Skipping batch 17/177, already exists
Skipping batch 18/177, already exists
Skipping batch 19/177, already exists
Skipping batch 20/177, already exists
Skipping batch 21/177, already exists
Skipping batch 22/177, already exists
Skipping batch 23/177, already exists
Skipping batch 24/177, already exists
Skipping batch 25/177, already exists
Skipping batch 26/177, already exists
Skipping batch 27/177

# Combine them all

In [16]:
all_annotations = {}

for file in sorted(output_dir.glob("autosave_results_*.json")):
    with open(file, "r", encoding="utf-8") as f:
        batch_data = json.load(f)

    all_annotations.update(batch_data)

with open(f"topic_annotations_WP{legislatur}.json", "w", encoding="utf-8") as f:
    json.dump(all_annotations, f, ensure_ascii=False, indent=2)

print(len(all_annotations))

8824


In [ ]:
themen_annotation_batch = {}
for rede, result in zip(reden_subset, results):
    flat = list(chain.from_iterable(result))
    counter = Counter(flat)
    total = sum(counter.values())
    counter = {
        k: v * num_topics / total
        for k, v in counter.items()
    }
    sorted_topics = dict(
        sorted(counter.items(), key=lambda x: x[1], reverse=True)
    )
    themen_annotation_batch[rede['id']] = sorted_topics
        

In [ ]:
themen_annotation_batch